# 05 – Feature Engineering: Variables Educativas

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Crear variables derivadas del nivel educativo que mejoren la capacidad predictiva del modelo sobre la condición de desocupación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FE_DIR = os.path.join('..', 'data', 'feature_engineering')
try:
    df = pd.read_csv(os.path.join(FE_DIR, 'epen_fe_employment.csv'))
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'edad': np.random.randint(14, 70, n),
        'nivel_educativo_ord': np.random.randint(0, 6, n),
        'target_desocupado': np.random.choice([0, 1], n, p=[0.50, 0.50]),
    })

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 1. Indicador de educación superior (Universidad o Posgrado)

In [ ]:
# nivel_educativo_ord: 0=Sin instrucción, 1=Primaria, 2=Secundaria,
#                      3=Preparatoria, 4=Universidad, 5=Posgrado
if 'nivel_educativo_ord' in df.columns:
    df['educacion_superior'] = (df['nivel_educativo_ord'] >= 4).astype(int)
    print('Variable creada: educacion_superior')
    print(df['educacion_superior'].value_counts())

## 2. Indicador de educación mínima obligatoria (Secundaria completa)

In [ ]:
if 'nivel_educativo_ord' in df.columns:
    df['edu_minima_obligatoria'] = (df['nivel_educativo_ord'] >= 2).astype(int)
    print('Variable creada: edu_minima_obligatoria')
    print(df['edu_minima_obligatoria'].value_counts())

## 3. Relación edad–educación

Detecta si la persona ha completado un nivel educativo razonable para su edad (proxy de trayectoria educativa regular).

In [ ]:
if 'edad' in df.columns and 'nivel_educativo_ord' in df.columns:
    # Años mínimos esperados de escolaridad según edad aproximada
    # 14-17: al menos secundaria (2); 18-22: preparatoria (3); 23+: universidad (4)
    def nivel_esperado(edad):
        if edad < 18:
            return 2
        elif edad < 23:
            return 3
        else:
            return 4

    expected = df['edad'].apply(nivel_esperado)
    df['trayectoria_regular'] = (df['nivel_educativo_ord'] >= expected).astype(int)
    print('Variable creada: trayectoria_regular')
    print(df['trayectoria_regular'].value_counts())

## 4. Visualización: Tasa de desocupación por nivel educativo

In [ ]:
if 'nivel_educativo_ord' in df.columns and 'target_desocupado' in df.columns:
    nivel_labels = {0: 'Sin instrucción', 1: 'Primaria', 2: 'Secundaria',
                    3: 'Preparatoria', 4: 'Universidad', 5: 'Posgrado'}
    tasa = df.groupby('nivel_educativo_ord')['target_desocupado'].mean()
    tasa.index = tasa.index.map(nivel_labels)
    tasa.plot(kind='bar', color='coral', edgecolor='black', figsize=(10, 4))
    plt.title('Tasa de desocupación por nivel educativo')
    plt.ylabel('Tasa')
    plt.xlabel('')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
os.makedirs(FE_DIR, exist_ok=True)
df.to_csv(os.path.join(FE_DIR, 'epen_fe_education.csv'), index=False)
nuevas = ['educacion_superior', 'edu_minima_obligatoria', 'trayectoria_regular']
print('Nuevas variables educativas:', [c for c in nuevas if c in df.columns])
print('Dataset guardado: epen_fe_education.csv')